In [2]:
import mediapipe as mp
print(mp.__version__)

1.2.0


In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, json, cv2, numpy as np, pandas as pd
import mediapipe as mp
from tqdm.auto import tqdm

BASE = '/content/gdrive/MyDrive/slr_project'
VIDEO_DIR = f'{BASE}/videos100'
OUT_DIR = f'{BASE}/keypoints'                 # output, one .npy per video
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(f'{BASE}/wlasl100_manifest.csv', dtype={'video_id': str})
df = df[df.status == 'ok'].reset_index(drop=True)
print(len(df), 'videos to process')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
1013 videos to process


In [4]:
print(mp.__version__)

1.2.0


In [5]:
!mkdir -p /content/videos100
!cp -r {VIDEO_DIR}/. /content/videos100/
LOCAL_VIDEOS = '/content/videos100'

In [6]:
def read_video(path):
    """Return list of BGR frames, or None if the video cannot be read."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames if len(frames) > 0 else None

In [7]:
SEQ_LEN = 32

def select_frames(frames, start, end, n=SEQ_LEN):
    total = len(frames)
    s = int(start) - 1 if pd.notna(start) and start > 0 else 0      # frame_start is 1-indexed
    e = int(end) - 1 if pd.notna(end) and end > 0 else total - 1    # end <= 0 means "until the end"
    s = max(0, min(s, total - 1))
    e = max(s, min(e, total - 1))
    idx = np.linspace(s, e, n).astype(int)                          # evenly spaced; repeats frames if clip is short
    return [frames[i] for i in idx]

In [8]:
MODEL_PATH = f'{BASE}/holistic_landmarker.task'
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task
print(round(os.path.getsize(MODEL_PATH) / 1e6, 1), 'MB')

13.7 MB


In [9]:
def make_landmarker():
    options = mp.tasks.vision.HolisticLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_hand_landmarks_confidence=0.5)
    return mp.tasks.vision.HolisticLandmarker.create_from_options(options)

def lm_array(lms, n, with_vis):
    """Flatten a landmark list to a vector; zeros if not detected."""
    if lms and isinstance(lms[0], (list, tuple)):      # some versions wrap it in an outer list
        lms = lms[0]
    if not lms or len(lms) != n:
        return np.zeros(n * (4 if with_vis else 3))
    if with_vis:
        return np.array([[l.x, l.y, l.z, l.visibility or 0.0] for l in lms]).flatten()
    return np.array([[l.x, l.y, l.z] for l in lms]).flatten()

def extract_keypoints(result):
    pose = lm_array(result.pose_landmarks, 33, True)           # 132
    lh = lm_array(result.left_hand_landmarks, 21, False)       # 63
    rh = lm_array(result.right_hand_landmarks, 21, False)      # 63
    return np.concatenate([pose, lh, rh])                       # 258

def process_frames(frames):
    feats = []
    with make_landmarker() as landmarker:          # fresh tracker per video, so videos don't influence each other
        for i, f in enumerate(frames):
            rgb = np.ascontiguousarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(mp_img, i * 40)   # timestamps must keep increasing
            feats.append(extract_keypoints(result))
    return np.array(feats)                                           # (SEQ_LEN, 258)

def missing_rate(feats):
    lh = (feats[:, 132:195] == 0).all(axis=1).mean()
    rh = (feats[:, 195:258] == 0).all(axis=1).mean()
    return lh, rh

In [10]:
r = df.iloc[0]
frames = read_video(f'{LOCAL_VIDEOS}/{r.video_id}.mp4')
feats = process_frames(select_frames(frames, r.frame_start, r.frame_end))
print(r.gloss, feats.shape, missing_rate(feats))

orange (32, 258) (np.float64(1.0), np.float64(0.3125))


In [11]:
log = []
for _, r in tqdm(df.iterrows(), total=len(df)):
    out_path = f'{OUT_DIR}/{r.video_id}.npy'
    if os.path.exists(out_path):
        continue
    try:
        frames = read_video(f'{LOCAL_VIDEOS}/{r.video_id}.mp4')
        if frames is None:
            log.append((r.video_id, 'unreadable')); continue
        sel = select_frames(frames, r.frame_start, r.frame_end)
        feats = process_frames(sel)
        np.save(out_path, feats.astype(np.float32))
        lh, rh = missing_rate(feats)
        log.append((r.video_id, 'ok', round(lh, 2), round(rh, 2)))
    except Exception as e:
        log.append((r.video_id, f'error: {e}'))

pd.DataFrame(log).to_csv(f'{BASE}/extraction_log.csv', index=False)

  0%|          | 0/1013 [00:00<?, ?it/s]

In [13]:
log_df = pd.read_csv(f'{BASE}/extraction_log.csv', header=None)
print(log_df[1].value_counts())       # how many ok / unreadable / error
done = len([f for f in os.listdir(OUT_DIR) if f.endswith('.npy')])
print(done, 'keypoint files of', len(df))

1
ok    477
1       1
Name: count, dtype: int64
1013 keypoint files of 1013


In [14]:
def normalize(feats):
    """feats: (T, 258). Center on shoulder midpoint and scale by shoulder width."""
    f = feats.copy()
    pose = f[:, :132].reshape(-1, 33, 4)
    ls, rs = pose[:, 11, :3], pose[:, 12, :3]            # left/right shoulder (x, y, z)
    center = (ls + rs) / 2                                # (T, 3)
    scale = np.linalg.norm(ls[:, :2] - rs[:, :2], axis=1)  # shoulder width in x-y plane
    scale[scale < 1e-3] = 1.0                             # avoid dividing by zero when pose is missing

    def adjust(block, n_points, has_vis):
        pts = block.reshape(-1, n_points, 4 if has_vis else 3)
        valid = (pts[:, :, :3] != 0).any(axis=2, keepdims=True)    # don't shift zeros (missing points)
        pts[:, :, :3] = np.where(valid, (pts[:, :, :3] - center[:, None, :]) / scale[:, None, None], 0)
        return pts.reshape(len(block), -1)

    out = np.concatenate([adjust(f[:, :132], 33, True),
                          adjust(f[:, 132:195], 21, False),
                          adjust(f[:, 195:258], 21, False)], axis=1)
    return out

In [15]:
classes = sorted(df['gloss'].unique())
label_map = {g: i for i, g in enumerate(classes)}
np.save(f'{BASE}/actions.npy', np.array(classes))

def build(split):
    sub = df[(df.subset == split)]
    X, y, vids, signers = [], [], [], []
    for _, r in sub.iterrows():
        p = f'{OUT_DIR}/{r.video_id}.npy'
        if not os.path.exists(p):
            continue
        X.append(np.load(p)); y.append(label_map[r.gloss])
        vids.append(r.video_id); signers.append(r.signer_id)
    return np.array(X), np.array(y), np.array(vids), np.array(signers)

for split in ['train', 'val', 'test']:
    X, y, vids, signers = build(split)
    np.savez_compressed(f'{BASE}/processed_{split}.npz', X=X, y=y, video_ids=vids, signer_ids=signers)
    print(split, X.shape, y.shape)

train (748, 32, 258) (748,)
val (165, 32, 258) (165,)
test (100, 32, 258) (100,)


In [16]:
d = np.load(f'{BASE}/processed_train.npz')
X = d['X']
print('NaNs:', np.isnan(X).sum())
print('all-zero videos:', (np.abs(X).sum(axis=(1, 2)) == 0).sum())
print('mean fraction of zero frames:', (np.abs(X).sum(axis=2) == 0).mean().round(3))

NaNs: 0
all-zero videos: 0
mean fraction of zero frames: 0.008
